### Structured output
- Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring   the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

#### Pydantic
- Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [6]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CCF4F76350>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CCF4F76D50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [8]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CCF4F76350>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CCF4F76D50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [9]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception**  \n*2010 American science‑fiction heist thriller*  \n\n| Item | Details |\n|------|---------|\n| **Director** | Christopher\u202fNolan |\n| **Screenplay** | Christopher\u202fNolan (adapted from a story by Nolan & Jonathan Nolan) |\n| **Producer(s)** | Emma Thomas, Christopher\u202fNolan, Christopher\u202fMiller |\n| **Music** | Hans\u202fZimmer |\n| **Cinematography** | Wally\u202fPfister |\n| **Studio** | Warner Bros. Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Release dates** | 16\u202fJuly\u202f2010 (USA – limited), 16\u202fJuly\u202f2010 (UK) |\n| **Runtime** | 148\u202fmin |\n| **Country** | United States |\n| **Language** | English |\n| **Budget** | ~\u202f$160\u202fmillion |\n| **Box‑office** | $829\u202fmillion worldwide |\n\n---\n\n### Premise\n\nA skilled “extractor” named **Dom Cobb** (Leonardo DiCaprio) steals secrets from the subconscious by infiltrating people’s dreams. He is offered a chance to have his criminal record er

In [10]:
response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Message output alongside parsed structure

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function to get movie details.', 'tool_calls': [{'id': 'fc_8921f31e-87d4-4e3e-a1a5-e136dfbebffe', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 164, 'total_tokens': 215, 'completion_time': 0.053061648, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.009425628, 'prompt_tokens_details': None, 'queue_time': 0.332939557, 'total_time': 0.062487276}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66f3850a1a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e26d-3e22-7781-ab3e-2832b85c9f5d-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'i

#### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

In [13]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Tom Berenger', role='Malcolm')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [14]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [15]:
model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [18]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

In [20]:
model_with_typeddict_nested = model.with_structured_output(MovieDetails)

response = model_with_typeddict_nested.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'The Avengers',
 'year': 2012}

In [22]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [23]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='7ce09883-7ea2-4c7c-b628-48dad83dd13e'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', 'extras': {'signature': 'EpIKCo8KAWkUfRPJ0slkxc0AB+/0PR6DowbEtTJ42jotAHaek5W/NCJtODIIn2TkaRpYC3vvN1panNXVqrdPmsus3kQKsR+jRQKgis60s7/q7FilR7jQnOd9JvRxY5O5yBu2b/mtjsTo5IxG8ieZrhzdEePyev9TlJOW+iSktYg48xp+nW92ttvozqFYcGnlX8t4S6zDI7ADma/Y48CuEP2K29kfodTF9lrj0FvczddnFyM3Mj+ptDV+B2TyDURUK4GdACPTO7KjirHzSyMd+qSNSDvmkiyxcNBiX7mcbtfvems7CvKJDw08+tq4C4KGZkAjc7GiDGwlexfYIwmsWnXIchOnyIKPOU/JV2kXYBZJzIGXRLrVLL9GrNUYrPxTr+U4slaxiTE+LgIiUmV8cn9VvSsAMSDcqy0GquDFpuuZJ4dZ5ew/2Hyw2d6EtHkG1xuir09y8oHeKjYakheYQKxttkC4mHBPJPmuCD6quY0bFTugTTxGrv1ro8fcT+0O0WbfuzmB3iqNQzJq5y0n3uovpkMsPHtjyRndrr5P8i0ulVlriwR7Rk4iBDPNp1AZlhqxTTdroMj3hNWwb/IA/DVn9YsQyFbznZc/+Y0wgiJOdiit8mH0+kqN67jkg1/cd4g8EGAzrzJ1WFvWQv

In [35]:
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [ ]:
## DataClas

In [39]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)




In [41]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
